In [3]:
# Clase abstracta Problem
class Problem:
    def __init__(self, initial, goal):
        self.initial = initial  # Estado inicial
        self.goal = goal        # Meta

    def actions(self, state):
        raise NotImplementedError

    # Función de transición
    def result(self, state, action):
        raise NotImplementedError

    # Función de desempeño
    def is_goal(self, state):
        return self.goal == state

    def action_cost(self, state1, action, state2):
        return 1

    def h(self, state):
        return 0


# Especialización del problema para grafos
class GraphAStartProblem(Problem):
    def __init__(self, initial, goal, graph):
        super().__init__(initial, goal)
        self.graph = graph

    def actions(self, state):
        return list(self.graph[state].keys())

    # Función de transición
    def result(self, state, action):
        return action

    def action_cost(self, state1, action, state2):
        return self.graph[state1][state2]

    def h(self, state):
        return straight_line_distance[state]


# Clase Node
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def path(self):
        lista_path = []
        node = self
        while node:
            lista_path.append(node.state)
            node = node.parent
        return lista_path[::-1]

    def expand(self, problem):
        lista = []
        for action in problem.actions(self.state):
            lista.append(self.child_node(problem, action))
        return lista

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        step_cost = problem.action_cost(self.state, action, next_state)
        return Node(next_state, self, action, self.path_cost + step_cost)

In [2]:
romania = {
    'Arad': {'Zerind': 75, 'Sibiu': 140, 'Timisoara': 118},
    'Zerind': {'Arad': 75, 'Oradea': 71},
    'Oradea': {'Zerind': 71, 'Sibiu': 151},
    'Sibiu': {'Arad': 140, 'Oradea': 151, 'Fagaras': 99, 'Rimnicu Vilcea': 80},
    'Timisoara': {'Arad': 118, 'Lugoj': 111},
    'Lugoj': {'Timisoara': 111, 'Mehadia': 70},
    'Mehadia': {'Lugoj': 70, 'Dobreta': 75},
    'Dobreta': {'Mehadia': 75, 'Craiova': 120},
    'Craiova': {'Dobreta': 120, 'Rimnicu Vilcea': 146, 'Pitesti': 138},
    'Rimnicu Vilcea': {'Sibiu': 80, 'Craiova': 146, 'Pitesti': 97},
    'Fagaras': {'Sibiu': 99, 'Bucarest': 211},
    'Pitesti': {'Rimnicu Vilcea': 97, 'Craiova': 138, 'Bucarest': 101},
    'Bucarest': {'Fagaras': 211, 'Pitesti': 101, 'Giurgiu': 90, 'Urziceni': 85},
    'Giurgiu': {'Bucarest': 90},
    'Urziceni': {'Bucarest': 85, 'Hirsova': 98, 'Vaslui': 142},
    'Hirsova': {'Urziceni': 98, 'Eforie': 86},
    'Eforie': {'Hirsova': 86},
    'Vaslui': {'Urziceni': 142, 'Iasi': 92},
    'Iasi': {'Vaslui': 92, 'Neamt': 87},
    'Neamt': {'Iasi': 87},
}

straight_line_distance = {
    'Arad': 366,
    'Bucarest': 0,
    'Craiova': 160,
    'Dobreta': 242,
    'Eforie': 161,
    'Fagaras': 178,
    'Giurgiu': 77,
    'Hirsova': 151,
    'Iasi': 226,
    'Lugoj': 244,
    'Mehadia': 241,
    'Neamt': 234,
    'Oradea': 380,
    'Pitesti': 98,
    'Rimnicu Vilcea': 193,
    'Sibiu': 253,
    'Timisoara': 329,
    'Urziceni': 80,
    'Vaslui': 199,
    'Zerind': 374,
}

In [5]:

def hill_climbing(problem):
    """ Búsqueda local informada (descenso de h). """
    current = Node(problem.initial)  # estado actual (Arad)

    while True:
        # TODO 1: current es la meta ?
        if problem.is_goal(current.state):
            return current, True

        # TODO 2: vecinos
        neighbors = current.expand(problem)
        if not neighbors:
            return current, False

        # TODO 3: vecino con MENOR h
        mejor = min(neighbors, key=lambda node: problem.h(node.state))

        # TODO 4: sin mejora => cima local
        if problem.h(mejor.state) >= problem.h(current.state):
            return current, False

        current = mejor

In [7]:

problem = GraphAStartProblem("Arad", "Bucarest", romania)

solution_node, success = hill_climbing(problem)

print("Camino encontrado:", solution_node.path())
print("Costo del camino;", solution_node.path_cost)

Camino encontrado: ['Arad', 'Sibiu', 'Fagaras', 'Bucarest']
Costo del camino (g): 450
